# Download HuggingFace datasets

Generic helper for fetching dataset repositories from the HuggingFace Hub. Downloads land
under `HF_HOME` (set below); this notebook then converts each shipped HF parquet shard into
the loose `images/<UID>.png` + `annotation.csv` layout that RadHarmony's harmonizer/dataset
classes expect.

> **Note.** A plain Python script (`.py`) is a perfectly valid alternative to this notebook if you prefer — e.g. run `python notebooks/download_<dataset>.py` from CI or a shell. Use whichever fits your workflow; the logic is identical.

## Prerequisites

1. `pip install huggingface_hub pyarrow` (auto-installed below if missing).
2. **HuggingFace token** (optional). Public datasets like TAIX-Ray (CC BY 4.0) work unauthenticated. For gated datasets, create a token at <https://huggingface.co/settings/tokens> and either export `HF_TOKEN` or paste it when prompted.
3. For gated datasets, accept the licence on the dataset page first.

In [ ]:
import getpass
import os
from pathlib import Path

# Where downloads land. Defaults to ~/path/to; override by exporting HF_HOME before launching Jupyter.
DATASETS_DIR = Path.home() / "path/to"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(DATASETS_DIR / ".hf_cache")

# Optional auth: only needed for gated/private datasets.
if not os.environ.get("HF_TOKEN"):
    raw = getpass.getpass("Paste HuggingFace token (hf_...) or press Enter to skip for public datasets: ")
    if raw.strip():
        os.environ["HF_TOKEN"] = raw.strip()

print("Downloads will be cached under:", DATASETS_DIR)
print("HF_HOME:", os.environ["HF_HOME"])
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Install dependencies into the active kernel if missing.
try:
    import huggingface_hub  # noqa: F401
    import pyarrow  # noqa: F401
except ImportError:
    get_ipython().run_line_magic('pip', 'install -q huggingface_hub pyarrow')
    import huggingface_hub  # noqa: F401
    import pyarrow  # noqa: F401

print("huggingface_hub:", huggingface_hub.__version__)
print("pyarrow:", pyarrow.__version__)

## Helper: parquet → images/&lt;UID&gt;.png + annotation.csv

HF datasets that ship images inside parquet shards (e.g. TAIX-Ray) need to be unpacked into
loose files for RadHarmony's MONAI-backed dataset classes. The function below:

1. Streams each parquet shard with PyArrow.
2. Writes `Image.bytes` verbatim to `dest_dir/images/<UID>.png` (preserves 16-bit fidelity — no PIL re-encoding).
3. Accumulates the non-image columns into a single `annotation.csv` at `dest_dir/annotation.csv`.

Resumable: existing PNGs are skipped, so re-running the cell is safe and cheap.

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd
from tqdm import tqdm


def parquet_dir_to_images_and_csv(
    parquet_dir: Path,
    dest_dir: Path,
    image_col: str = "Image",
    uid_col: str = "UID",
) -> Path:
    """Unpack every parquet shard in `parquet_dir` to `dest_dir/images/<UID>.png` plus `dest_dir/annotation.csv`.

    Returns the path to the written annotation.csv.
    """
    parquet_dir = Path(parquet_dir)
    dest_dir = Path(dest_dir)
    images_dir = dest_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    shards = sorted(parquet_dir.glob("*.parquet"))
    if not shards:
        raise FileNotFoundError(f"No .parquet files found under {parquet_dir}")

    metadata_frames: list[pd.DataFrame] = []
    for shard in tqdm(shards, desc=f"shards ({parquet_dir.name})"):
        table = pq.read_table(shard)
        non_image_cols = [c for c in table.schema.names if c != image_col]
        meta = table.select(non_image_cols).to_pandas()
        metadata_frames.append(meta)

        uids = table.column(uid_col).to_pylist()
        images = table.column(image_col).to_pylist()
        for uid, img in zip(uids, images):
            out = images_dir / f"{uid}.png"
            if out.exists():
                continue
            png_bytes = img["bytes"] if isinstance(img, dict) else img
            out.write_bytes(png_bytes)

    annotations = pd.concat(metadata_frames, ignore_index=True)
    csv_path = dest_dir / "annotation.csv"
    annotations.to_csv(csv_path, index=False)
    print(f"Wrote {len(annotations):,} rows to {csv_path}")
    print(f"PNG count under {images_dir}: {sum(1 for _ in images_dir.glob('*.png')):,}")
    return csv_path

## Download: TAIX-Ray (512-pixel config)

~58 GB. 215,381 bedside chest radiographs, longer dimension standardized to 512 px.
Used by `TAIXRay512Dataset`. Public dataset — no token required.

<https://huggingface.co/datasets/TLAIM/TAIX-Ray>

In [ ]:
from huggingface_hub import snapshot_download

REPO_ID = "TLAIM/TAIX-Ray"
TAIX_ROOT = DATASETS_DIR / "TAIX-Ray"
TAIX_512_DIR = TAIX_ROOT / "data_512"

snapshot_path = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=["data/*.parquet", "README.md"],
    local_dir=TAIX_ROOT / ".snapshot_512",
)
print("Snapshot at:", snapshot_path)

In [ ]:
parquet_dir_to_images_and_csv(
    parquet_dir=Path(snapshot_path) / "data",
    dest_dir=TAIX_512_DIR,
)

## Download: TAIX-Ray (original-resolution config)

~1.27 TB. Same 215,381 radiographs at native acquisition resolution. Used by
`TAIXRayDataset`. Skip this cell if you only need the 512-px version.

In [ ]:
TAIX_ORIGINAL_DIR = TAIX_ROOT / "data_original"

snapshot_path_orig = snapshot_download(
    repo_id=REPO_ID,
    repo_type="dataset",
    allow_patterns=["original/*.parquet", "README.md"],
    local_dir=TAIX_ROOT / ".snapshot_original",
)
print("Snapshot at:", snapshot_path_orig)

In [ ]:
parquet_dir_to_images_and_csv(
    parquet_dir=Path(snapshot_path_orig) / "original",
    dest_dir=TAIX_ORIGINAL_DIR,
)

## Make dataset readable to other users

After download + conversion, relax permissions on the dataset root so collaborators in the
same group can read the files. `775` = owner+group rwx, others rx.

In [ ]:
import subprocess

subprocess.run(["chmod", "-R", "775", str(TAIX_ROOT)], check=True)
print("Permissions updated under:", TAIX_ROOT)